# 📖 Notebook 3: Distributed Top-K

In Notebooks 1 and 2, we built Count-Min Sketch and a Heap-based Top-K — both running on a **single machine**. But YouTube processes ~700K view events per second. No single machine can handle that.

In this notebook, we'll explore the patterns used to distribute Top-K across multiple machines:
- **Sharding** — split the work across partitions
- **Tumbling windows** — aggregate counts into time buckets
- **Redis sorted sets** — a shared data structure for real-time leaderboards
- **Precomputation + caching** — compute once, serve many times

## Learning Objectives

By the end of this notebook, you'll understand:
- Why a single machine can't handle YouTube-scale view counting
- How sharding splits the work and how to merge partial top-K results
- What tumbling windows are and how they simplify time-based queries
- How Redis sorted sets work as a real-time leaderboard
- How precomputation + caching makes queries fast

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/top-k
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `topk_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import json
import heapq
import hashlib
import random
from datetime import datetime, timedelta
from collections import defaultdict

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "topk_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 Why One Machine Isn't Enough

Let's do some quick math to understand the scale:

- YouTube: ~70 billion views/day = ~700K views/second
- A single PostgreSQL can handle ~10K writes/second under ideal conditions
- 700K / 10K = **70 database instances** just for writes!

And that's just counting. We also need to:
- Store view events (or at least hourly aggregates)
- Query the top K across ALL videos for different time windows
- Serve results in <50ms to users

We need to **split the work** across multiple machines.

In [ ]:
print("📊 Scale Estimation: YouTube Top-K")
print("=" * 55)
print()

views_per_day = 70_000_000_000
seconds_per_day = 100_000  # rounded for easy math (actual: 86,400)
views_per_second = views_per_day // seconds_per_day

print(f"  Views per day:     {views_per_day:>15,}")
print(f"  Views per second:  {views_per_second:>15,}")
print()

# Storage estimation
videos_per_day = 1_000_000  # ~12 uploads/second, sustained
total_videos = videos_per_day * 365 * 10  # 10 years
bytes_per_entry = 16  # 8 bytes video_id + 8 bytes count — the DATA only
naive_storage_gb = total_videos * bytes_per_entry / (1024**3)

print(f"  Total videos (10yr): {total_videos:>13,}")
print(f"  Naive storage:       {naive_storage_gb:>13.0f} GB  (16 B/entry, data only)")
print(f"  ...in a Python dict: {total_videos * 80 / (1024**3):>13.0f} GB  (80 B/entry, measured in NB1)")
print()

# Sharding needs
db_writes_per_sec = 10_000
shards_needed = views_per_second // db_writes_per_sec
print(f"  DB capacity:         {db_writes_per_sec:>13,} writes/sec")
print(f"  Shards needed:       {shards_needed:>13,}")
print()
print("💡 We need ~70 DB shards just for writes — and that's BEFORE")
print("   considering that window queries scan millions of rows!")

## 🔀 Pattern 1: Sharding

**Sharding** splits data across multiple machines based on a key (video ID).  
Each shard handles a subset of videos independently.

```
  View Events (Kafka, partitioned by video_id)
  ┌──────────┬──────────┬──────────┐
  │ Shard 0  │ Shard 1  │ Shard 2  │
  │ vid_000  │ vid_001  │ vid_002  │
  │ vid_003  │ vid_004  │ vid_005  │
  │ vid_006  │ vid_007  │ vid_008  │
  │   ...    │   ...    │   ...    │
  └────┬─────┴────┬─────┴────┬─────┘
       │          │          │
       ▼          ▼          ▼
  ┌─────────┐┌─────────┐┌─────────┐
  │  DB 0   ││  DB 1   ││  DB 2   │
  └────┬────┘└────┬────┘└────┬────┘
       │          │          │
       └──────────┼──────────┘
                  │
            ┌─────▼─────┐
            │  MERGE    │  ← Get top K from each shard,
            │  Top-K    │    merge into global top K
            └───────────┘
```

**The key insight — and its precondition**: if you get the top K from EACH
shard, the global top K is somewhere in the union of those partial results —
**but only because we shard by `video_id`.**

That choice means every view of a video lands on the same shard, so a shard's
local count for a video *is* its global count. A video in the global top K
therefore cannot be outside its own shard's top K: at most K−1 videos beat it
globally, so at most K−1 beat it locally.

Break that assumption — shard by ingest server, by region, by user — and the
merge silently returns the wrong answer. We reproduce exactly that failure
two cells from here, so you can see what it looks like.

In [ ]:
# Load view events from DB
conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT video_id, viewed_at FROM view_events ORDER BY viewed_at")
all_events = cursor.fetchall()
conn.close()

print(f"📦 Loaded {len(all_events):,} view events")
print()

# Simulate sharding: assign each video to a shard based on hash
NUM_SHARDS = 3

def get_shard(video_id: str, num_shards: int) -> int:
    """Determine which shard a video belongs to using consistent hashing."""
    hash_val = int(hashlib.md5(video_id.encode()).hexdigest(), 16)
    return hash_val % num_shards

# Split events into shards
shard_counts = [defaultdict(int) for _ in range(NUM_SHARDS)]
for video_id, viewed_at in all_events:
    shard = get_shard(video_id, NUM_SHARDS)
    shard_counts[shard][video_id] += 1

for i in range(NUM_SHARDS):
    total = sum(shard_counts[i].values())
    print(f"  Shard {i}: {len(shard_counts[i]):>3} videos, {total:>6,} events")

print()
print("💡 Each shard only handles a subset of videos.")
print("   This spreads the write load across multiple databases.")

In [ ]:
# Merge: get top K from each shard, then merge
K = 10

print(f"🔀 Shard-Local Top-{K} → Merge → Global Top-{K}")
print("=" * 60)

# Step 1: Get top K from each shard
shard_top_k = []
for i in range(NUM_SHARDS):
    local_top = sorted(
        shard_counts[i].items(), key=lambda x: x[1], reverse=True
    )[:K]
    shard_top_k.append(local_top)
    
    print(f"\n  Shard {i} top {K}:")
    for rank, (vid, count) in enumerate(local_top[:5], 1):
        print(f"    {rank}. {vid}: {count:,} views")
    if len(local_top) > 5:
        print(f"    ... and {len(local_top) - 5} more")

# Step 2: Merge all shard results and take global top K
all_candidates = []
for local_top in shard_top_k:
    all_candidates.extend(local_top)

global_top_k = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:K]

print(f"\n🏆 Global Top {K} (merged from {NUM_SHARDS} shards):")
print(f"  {'Rank':<6} {'Video ID':<12} {'Views':>8}")
print("  " + "-" * 28)
for rank, (vid, count) in enumerate(global_top_k, 1):
    print(f"  {rank:<6} {vid:<12} {count:>8,}")

# This is only safe because get_shard() keys on video_id. Prove it rather than
# asserting it in prose: the merged answer must match a single global count.
global_counts = defaultdict(int)
for video_id, _ in all_events:
    global_counts[video_id] += 1
true_global = sorted(global_counts.items(), key=lambda x: x[1], reverse=True)[:K]

# A video's shard-local count must equal its global count under key sharding.
for video_id, count in global_counts.items():
    local = shard_counts[get_shard(video_id, NUM_SHARDS)][video_id]
    assert local == count, (
        f"{video_id} has {local} views on its shard but {count} globally — "
        f"key-based sharding is supposed to keep an item whole"
    )

kth_count = true_global[-1][1]
unambiguous = {vid for vid, c in global_counts.items() if c > kth_count}
merged_ids = {vid for vid, _ in global_top_k}
assert unambiguous <= merged_ids, (
    f"shard merge lost {sorted(unambiguous - merged_ids)} — "
    f"impossible when sharding by the counting key"
)
assert [c for _, c in global_top_k] == [c for _, c in true_global], (
    f"merged counts {[c for _, c in global_top_k]} != "
    f"global counts {[c for _, c in true_global]}"
)

print()
print("✅ Merging shard-local top-K results gives the correct global top-K")
print(f"   (verified against a single global count: all {len(unambiguous)} unambiguous")
print("    heavy hitters survived, and every merged count matches exactly).")
print("   Each shard runs its query independently → easy to parallelize!")

### ⚠️ Change the shard key and the merge breaks

The section above works. That makes it a dangerous thing to memorise, because
the correctness came from the *shard key*, not from the merging.

In real pipelines you do not always get to choose. Events arrive at whichever
ingest server the load balancer picked; a Kafka topic may be partitioned by
`user_id` because some other consumer needed it that way; a multi-region
deployment counts locally before it counts globally. In all of those, one
video's views are **split across shards**.

Now watch what happens to a video that is broadly popular everywhere but
number one nowhere — the *slow burn*. It is the single most-watched video on
the platform, and it appears in **no shard's local top-K**, so the merge never
even sees it.

Then we'll fix it the other way: by merging **sketches** instead of lists.
A Count-Min Sketch is a table of counters, so two sketches with the same
dimensions and seeds add element-wise, and the sum is exactly the sketch you
would have built from the combined stream. A top-K *list* has no such
property — it has already discarded everything below its local cut.


In [ ]:
# What breaks when a video's views are SPLIT across shards, and what fixes it.
#
# New setup: events land on whichever ingest server took the request — a Kafka
# partition keyed by user, a regional edge, a load balancer's pick. Nothing
# guarantees that one video's views all land in the same place.
import mmh3

random.seed(11)
NUM_INGEST = 6
MERGE_K = 5

split_events = []          # (video_id, ingest_server)

# One SLOW BURN: broadly popular, evenly spread — 200 views on each of 6 servers.
for i in range(1_200):
    split_events.append(("vid_slowburn", i % NUM_INGEST))

# Per server, K videos that are viral ONLY there — 300 views, all on one server.
for server in range(NUM_INGEST):
    for j in range(MERGE_K):
        for _ in range(300):
            split_events.append((f"vid_local_{server}_{j}", server))

# Long tail, sprayed everywhere.
for _ in range(3_000):
    split_events.append((f"vid_tail_{random.randrange(400):03d}",
                         random.randrange(NUM_INGEST)))
random.shuffle(split_events)

# Ground truth
true_counts = defaultdict(int)
for vid, _ in split_events:
    true_counts[vid] += 1
true_top = sorted(true_counts.items(), key=lambda x: x[1], reverse=True)[:MERGE_K]

# Per-shard exact counts
ingest_counts = [defaultdict(int) for _ in range(NUM_INGEST)]
for vid, server in split_events:
    ingest_counts[server][vid] += 1

print(f"📨 {len(split_events):,} events across {NUM_INGEST} ingest servers "
      f"(NOT partitioned by video_id)")
print(f"\n🎯 TRUE global top-{MERGE_K}:")
for rank, (vid, c) in enumerate(true_top, 1):
    print(f"   {rank}. {vid:<20} {c:>6,}")

# ---- ❌ Approach A: each shard reports its top-K, we merge the LISTS --------
print(f"\n❌ Each shard's local top-{MERGE_K} (note where the slow burn sits):")
naive_totals = defaultdict(int)
for server in range(NUM_INGEST):
    local = sorted(ingest_counts[server].items(),
                   key=lambda x: x[1], reverse=True)[:MERGE_K]
    for vid, c in local:
        naive_totals[vid] += c
    print(f"   shard {server}: local #{MERGE_K} has {local[-1][1]:>4} views | "
          f"vid_slowburn has only {ingest_counts[server]['vid_slowburn']:>4} here")

naive_top = sorted(naive_totals.items(), key=lambda x: x[1], reverse=True)[:MERGE_K]
print(f"\n   Merged top-{MERGE_K} from the shard LISTS:")
for rank, (vid, c) in enumerate(naive_top, 1):
    print(f"   {rank}. {vid:<20} {c:>6,}")

naive_ids = {vid for vid, _ in naive_top}
true_number_one = true_top[0][0]
print(f"\n   → the true #1 ({true_number_one}, {true_counts[true_number_one]:,} views) "
      f"is {'MISSING' if true_number_one not in naive_ids else 'present'}.")

assert true_number_one not in naive_ids, (
    "this cell exists to reproduce a failure and the failure did not happen — "
    "the slow-burn video should be crowded out of every shard's local top-K"
)

# ---- ✅ Approach B: merge the SKETCHES, then rank ---------------------------
# A Count-Min Sketch is just a table of counters, so two sketches with the same
# dimensions and seeds add element-wise. The sum is bit-for-bit the sketch you
# would have built from the combined stream — no information is lost.
class MergeableCMS:
    def __init__(self, width, depth):
        self.width, self.depth = width, depth
        self.table = [[0] * width for _ in range(depth)]

    def _hash(self, item, row):
        return mmh3.hash(item, seed=row) % self.width

    def add(self, item, count=1):
        for row in range(self.depth):
            self.table[row][self._hash(item, row)] += count

    def estimate(self, item):
        return min(self.table[r][self._hash(item, r)] for r in range(self.depth))

    def merge(self, other):
        """Element-wise add. Requires identical width, depth and seeds."""
        assert (self.width, self.depth) == (other.width, other.depth), \
            "sketches only merge if they were built with the same dimensions"
        out = MergeableCMS(self.width, self.depth)
        out.table = [[a + b for a, b in zip(ra, rb)]
                     for ra, rb in zip(self.table, other.table)]
        return out

W, D = 4_000, 5
shard_sketches = [MergeableCMS(W, D) for _ in range(NUM_INGEST)]
for vid, server in split_events:
    shard_sketches[server].add(vid)

merged = shard_sketches[0]
for s in shard_sketches[1:]:
    merged = merged.merge(s)

# The property that makes this work: merging is EXACTLY equivalent to having
# counted the whole stream in one sketch.
single = MergeableCMS(W, D)
for vid, _ in split_events:
    single.add(vid)
assert merged.table == single.table, (
    "merged sketch differs from a sketch built over the whole stream — "
    "merging is supposed to be lossless"
)

merged_top = sorted(((v, merged.estimate(v)) for v in true_counts),
                    key=lambda x: x[1], reverse=True)[:MERGE_K]
print(f"\n✅ Merged the {NUM_INGEST} SKETCHES instead, then ranked:")
for rank, (vid, c) in enumerate(merged_top, 1):
    print(f"   {rank}. {vid:<20} {c:>6,} (true {true_counts[vid]:,})")

for vid, c in true_counts.items():
    assert merged.estimate(vid) >= c, f"merged sketch undercounted {vid}"
assert merged_top[0][0] == true_number_one, (
    f"sketch merge should recover the true #1 ({true_number_one}), "
    f"got {merged_top[0][0]}"
)

print()
print("💡 The rule to remember:")
print("   • A SKETCH merges. Add the tables; the result is exact-as-if-combined.")
print("   • A TOP-K LIST does not merge. It has already thrown away every count")
print("     below the local cut, and that is precisely where the slow burn was.")
print()
print("   Two ways out, both used in production:")
print("   1. Partition by the counting key (what the section above does), so")
print("      each item's whole count lives on one shard and local top-K is")
print("      globally faithful. This is why Kafka topics are keyed by video_id.")
print("   2. Ship mergeable state — sketches, or per-shard counts for a")
print("      candidate set — and rank only after merging. Costs more network,")
print("      but survives any partitioning scheme.")


## ⏰ Pattern 2: Tumbling Windows

For time-based queries ("top videos in the last hour"), we need to organize events by time.

**Tumbling windows** divide time into fixed, non-overlapping buckets:

```
  Time: 9:00    10:00    11:00    12:00    13:00
        |--------|--------|--------|--------|
        Window 1  Window 2  Window 3  Window 4
        
  Each window is independent. No overlap.
  "Last hour" at 13:06 = Window 4 (13:00–14:00)
```

This is simpler than **sliding windows** (where "last hour" at 13:06 = 12:06–13:06)  
and is what our `hourly_views` table already uses.

In [ ]:
# Demonstrate tumbling windows using our view events

def truncate_to_hour(dt):
    """Truncate a datetime to the start of its hour (tumbling window)."""
    return dt.replace(minute=0, second=0, microsecond=0)

# Group events into hourly windows
hourly_windows = defaultdict(lambda: defaultdict(int))

for video_id, viewed_at in all_events:
    hour_bucket = truncate_to_hour(viewed_at)
    hourly_windows[hour_bucket][video_id] += 1

# Sort windows by time
sorted_windows = sorted(hourly_windows.keys())

print(f"⏰ Tumbling Windows: {len(sorted_windows)} hourly buckets")
print("=" * 60)

# Show a few windows
for window in sorted_windows[:5]:
    counts = hourly_windows[window]
    total_events = sum(counts.values())
    top_vid = max(counts.items(), key=lambda x: x[1])
    print(f"  {window.strftime('%Y-%m-%d %H:00')} │ {total_events:>5} events │"
          f" {len(counts):>3} videos │ top: {top_vid[0]} ({top_vid[1]})")

if len(sorted_windows) > 5:
    print(f"  ... ({len(sorted_windows) - 5} more windows)")

print()
print("💡 Each window can be processed independently.")
print("   To get 'last 24 hours': sum the last 24 windows.")
print("   To get 'last month': sum the last ~720 windows.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Visualize the tumbling windows
fig, ax = plt.subplots(figsize=(14, 5))

window_times = sorted_windows
window_totals = [sum(hourly_windows[w].values()) for w in window_times]

ax.bar(window_times, window_totals, width=0.03, color='steelblue', alpha=0.7)
ax.set_xlabel('Time (hourly buckets)')
ax.set_ylabel('View Events')
ax.set_title('View Events per Tumbling Window (1-hour buckets)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:00'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("💡 Each bar is one tumbling window (1 hour).")
print("   Events are fairly evenly distributed because our seed data")
print("   uses random timestamps within the last 48 hours.")

### ⏳ Tumbling vs. Sliding Windows

A **tumbling window** snaps to fixed boundaries (e.g., 12:00–13:00, 13:00–14:00).
A **sliding window** moves continuously with the current time (e.g., "the last
60 minutes from right now").

* ✅ Tumbling is cheap — every event belongs to exactly one bucket, and the
  bucket is fully done once its hour ends.
* ⚠️ Sliding is more accurate for "latest N minutes" queries but requires
  keeping raw events (or many tiny buckets) so you can re-sum on every query.

A common **compromise** in production: use **small tumbling buckets** (e.g.,
1-minute) and sum the most recent N of them to *approximate* a sliding window.
Let's show this trade-off directly on our data.


In [ ]:
from datetime import datetime, timedelta

# Pick a reference "now" that shows the trade-off instead of hiding it.
#
# The gap between tumbling and sliding is widest EARLY in an hour: the bucket
# has only had a few minutes to fill, while "last 60 minutes" reaches back into
# the previous bucket. Picking the newest event outright would land us at a
# random point in the hour — sometimes 2 minutes in (big gap), sometimes 58
# (no visible gap at all). So we deliberately pick the latest event that falls
# in the first quarter of its hour.
EARLY_MINUTES = 15
early_events = [ts for _, ts in all_events if ts.minute < EARLY_MINUTES]
assert early_events, (
    f"no event lands in the first {EARLY_MINUTES} min of an hour — cannot show "
    f"the tumbling/sliding gap with this data"
)
now_ref = max(early_events)

# ---- Tumbling: "this hour" = the bucket containing now_ref ----
tumbling_bucket = truncate_to_hour(now_ref)
tumbling_counts = defaultdict(int)
for vid, viewed_at in all_events:
    if tumbling_bucket <= viewed_at <= now_ref:
        tumbling_counts[vid] += 1

# ---- Sliding: "last 60 minutes up to now_ref" ----
sliding_cutoff = now_ref - timedelta(minutes=60)
sliding_counts = defaultdict(int)
for vid, viewed_at in all_events:
    if sliding_cutoff < viewed_at <= now_ref:
        sliding_counts[vid] += 1

def top_k(counts, k=5):
    return sorted(counts.items(), key=lambda x: x[1], reverse=True)[:k]

minutes_into_hour = (now_ref - tumbling_bucket).total_seconds() / 60

print(f"Reference 'now': {now_ref}  ({minutes_into_hour:.0f} min into its hour)")
print()
print(f"📊 Tumbling bucket {tumbling_bucket.strftime('%H:00')}–{(tumbling_bucket+timedelta(hours=1)).strftime('%H:00')}, "
      f"so far: {sum(tumbling_counts.values())} events, {len(tumbling_counts)} videos")
for vid, c in top_k(tumbling_counts):
    print(f"   {vid}: {c}")
print()
print(f"📊 Sliding window {sliding_cutoff.strftime('%H:%M')}–{now_ref.strftime('%H:%M')} "
      f"(exactly 60 min): {sum(sliding_counts.values())} events, {len(sliding_counts)} videos")
for vid, c in top_k(sliding_counts):
    print(f"   {vid}: {c}")

# The sliding window strictly contains the partial tumbling bucket, so it can
# never see fewer events — and here it must see strictly more, or there is no
# trade-off left to teach.
assert sum(sliding_counts.values()) > sum(tumbling_counts.values()), (
    f"sliding ({sum(sliding_counts.values())} events) did not beat the partial "
    f"tumbling bucket ({sum(tumbling_counts.values())}) at {minutes_into_hour:.0f} "
    f"min into the hour — the window comparison is not demonstrating anything"
)

print()
print(f"💡 'now' is only {minutes_into_hour:.0f} minutes into its tumbling bucket, so that")
print(f"   bucket has seen {sum(tumbling_counts.values())} events while the full 60-minute")
print(f"   sliding window has seen {sum(sliding_counts.values())}.")
print("   Tumbling undercounts early in the hour, then its answer jumps at the")
print("   boundary. Sliding is exact but must re-sum raw events on every query.")
print()
print("⚠️  Sample-size caveat: our seed data is ~488 events over 48 hours, so")
print("    these window counts are single digits and the top-5 lists are mostly")
print("    ties. The mechanism is real; the specific ranking here is noise.")
print("    Trust the event COUNTS above, not the ordering of the videos.")

## 🔴 Pattern 3: Redis Sorted Sets

Redis **sorted sets** (ZSET) are a perfect fit for leaderboards and top-K:
- Each member has a **score** (the view count)
- Members are always sorted by score
- Getting the top K is O(K + log N) — very fast!

In the distributed pipeline, Redis acts as the **shared cache** that all servers read from:

```
  View Consumers          PostgreSQL Shards         Redis
  ┌──────────┐           ┌──────────┐            ┌──────────────┐
  │ Consume  │──write──▶│  Count   │            │  ZSET:       │
  │ events   │           │  views   │            │  top-k:hour  │
  └──────────┘           └────┬─────┘            │  top-k:day   │
                              │                  │  top-k:all   │
                         ┌────▼─────┐            └──────▲───────┘
                         │ Top-K    │──precompute──────┘
                         │ Cron Job │
                         └──────────┘
```

In [ ]:
r = get_redis_client()

# Clean up any previous run
for key in r.keys("notebook3:*"):
    r.delete(key)

# Demonstrate Redis Sorted Sets for top-K
print("🔴 Redis Sorted Sets (ZSET) Basics")
print("=" * 50)

# ZADD: add members with scores
zset_key = "notebook3:demo_leaderboard"
r.zadd(zset_key, {
    "vid_010": 4000,
    "vid_001": 2500,
    "vid_021": 2250,
    "vid_003": 2000,
    "vid_036": 1750,
    "vid_007": 1250,
})

print("\n  Added 6 videos with view counts to a sorted set.")
print()

# ZREVRANGE: get top K (highest scores first)
top3 = r.zrevrange(zset_key, 0, 2, withscores=True)
print("  ZREVRANGE (top 3):")
for rank, (vid, score) in enumerate(top3, 1):
    print(f"    {rank}. {vid}: {int(score):,} views")
print()

# ZINCRBY: atomically increment a score
r.zincrby(zset_key, 500, "vid_001")  # vid_001: 2500 + 500 = 3000
print("  ZINCRBY vid_001 +500 → ", int(r.zscore(zset_key, "vid_001")))
print()

# ZREVRANK: what rank is a specific video?
rank = r.zrevrank(zset_key, "vid_003")
print(f"  ZREVRANK vid_003 → rank {rank} (0-indexed)")
print()

# ZCARD: how many members?
print(f"  ZCARD → {r.zcard(zset_key)} members")
print()
print("💡 All operations are O(log N) — fast even with millions of members.")
print("   ZREVRANGE for top K is O(K + log N) — perfect for leaderboards!")

In [ ]:
# Build a real top-K leaderboard from our view events
# This simulates what a view consumer would do in production

r = get_redis_client()
K = 10

# === All-time leaderboard ===
alltime_key = "notebook3:topk:alltime"
# ZINCRBY accumulates, so start clean — otherwise re-running this cell doubles
# every score and the exactness check below (rightly) fails.
r.delete(alltime_key)

# Use a pipeline for efficiency (batches commands to Redis)
start = time.time()
pipe = r.pipeline()
for video_id, viewed_at in all_events:
    pipe.zincrby(alltime_key, 1, video_id)
pipe.execute()
ingest_time = (time.time() - start) * 1000

# Query top K
start = time.time()
top_alltime = r.zrevrange(alltime_key, 0, K - 1, withscores=True)
query_time = (time.time() - start) * 1000

print(f"🏆 All-Time Top {K} — Redis Sorted Set")
print(f"   Ingested {len(all_events):,} events in {ingest_time:.0f} ms")
print(f"   Query time: {query_time:.2f} ms")
print("=" * 50)
print(f"  {'Rank':<6} {'Video ID':<12} {'Views':>8}")
print("  " + "-" * 28)
for rank, (vid, score) in enumerate(top_alltime, 1):
    print(f"  {rank:<6} {vid:<12} {int(score):>8,}")

print()
print(f"⚡ Query returned in {query_time:.2f} ms — well under 50ms target!")

assert query_time < 50, (
    f"ZREVRANGE took {query_time:.2f} ms, over the 50 ms target from the "
    f"requirements — the whole point of the sorted set is O(K + log N) reads"
)
# The leaderboard must agree with a plain count; a sorted set is exact.
zset_counts = {vid: int(score) for vid, score in top_alltime}
exact_counts = defaultdict(int)
for video_id, _ in all_events:
    exact_counts[video_id] += 1
for vid, score in zset_counts.items():
    assert score == exact_counts[vid], (
        f"ZSET score for {vid} is {score} but the true count is {exact_counts[vid]}"
    )
print(f"✅ All {len(zset_counts)} leaderboard scores match the exact counts "
      f"(ZSETs are not approximate).")

In [ ]:
# === Windowed leaderboards (last 24h, per-hour buckets) ===

# In production, view consumers write to per-hour sorted sets.
# Then a cron job merges them into the window leaderboard.

# Step 1: Populate per-hour sorted sets (clear first — ZINCRBY accumulates)
stale_hourly = r.keys("notebook3:hourly:*")
if stale_hourly:
    r.delete(*stale_hourly)

pipe = r.pipeline()
for video_id, viewed_at in all_events:
    hour_bucket = truncate_to_hour(viewed_at)
    hour_key = f"notebook3:hourly:{hour_bucket.strftime('%Y%m%d_%H')}"
    pipe.zincrby(hour_key, 1, video_id)
pipe.execute()

print("✅ Populated per-hour sorted sets in Redis")
print()

# Step 2: Merge the last 24 hours (simulating what a cron job would do)
# ZUNIONSTORE merges multiple sorted sets, summing scores
last_24h_keys = []
for window in sorted_windows[-24:]:
    hour_key = f"notebook3:hourly:{window.strftime('%Y%m%d_%H')}"
    if r.exists(hour_key):
        last_24h_keys.append(hour_key)

window_key = "notebook3:topk:last_24h"

start = time.time()
if last_24h_keys:
    r.zunionstore(window_key, last_24h_keys)
merge_time = (time.time() - start) * 1000

# Query the merged result
start = time.time()
top_24h = r.zrevrange(window_key, 0, K - 1, withscores=True)
query_time = (time.time() - start) * 1000

print(f"🏆 Last 24h Top {K} — Merged from {len(last_24h_keys)} hourly sets")
print(f"   Merge time: {merge_time:.2f} ms (ZUNIONSTORE)")
print(f"   Query time: {query_time:.2f} ms")
print("=" * 50)
print(f"  {'Rank':<6} {'Video ID':<12} {'Views':>8}")
print("  " + "-" * 28)
for rank, (vid, score) in enumerate(top_24h, 1):
    print(f"  {rank:<6} {vid:<12} {int(score):>8,}")

print()
print("💡 ZUNIONSTORE merges sorted sets server-side — very efficient!")
print("   This is what a Top-K cron job does every minute.")

## 🗄️ Pattern 4: Precomputation + Caching

The final piece: instead of computing top-K on every request, we **precompute** it periodically  
and store the result in a cache. Clients always read from cache.

```
  Every 1 minute:
  ┌─────────────┐     ┌──────────┐     ┌──────────────┐
  │  Cron Job   │────▶│ Compute  │────▶│ Cache result  │
  │  (periodic) │     │ top-K    │     │ in Redis      │
  └─────────────┘     └──────────┘     └──────┬───────┘
                                              │
                                              ▼
  ┌─────────────┐                     ┌──────────────┐
  │  Client:    │────GET /top-k──────▶│ Read from    │
  │  GET /top-k │◀───< 1ms ──────────│ cache only   │
  └─────────────┘                     └──────────────┘
```

Benefits:
- Client queries always hit cache → sub-millisecond latency
- Database is never hit by user queries
- Stale data is OK (within the 1-minute tolerance from our requirements)

In [ ]:
r = get_redis_client()

def precompute_top_k(window: str, k: int = 10):
    """
    Simulate a cron job that precomputes top-K for a given window
    and caches the result as a JSON string in Redis.
    
    In production, this runs every minute.
    """
    conn = get_db_connection()
    cursor = conn.cursor()
    
    if window == "alltime":
        cursor.execute("""
            SELECT vt.video_id, v.title, vt.total_views
            FROM video_view_totals vt
            JOIN videos v ON v.video_id = vt.video_id
            ORDER BY vt.total_views DESC
            LIMIT %s
        """, (k,))
    elif window == "last_hour":
        cursor.execute("""
            SELECT h.video_id, v.title, SUM(h.view_count) as total
            FROM hourly_views h
            JOIN videos v ON v.video_id = h.video_id
            WHERE h.hour_bucket >= NOW() - INTERVAL '1 hour'
            GROUP BY h.video_id, v.title
            ORDER BY total DESC
            LIMIT %s
        """, (k,))
    elif window == "last_day":
        cursor.execute("""
            SELECT h.video_id, v.title, SUM(h.view_count) as total
            FROM hourly_views h
            JOIN videos v ON v.video_id = h.video_id
            WHERE h.hour_bucket >= NOW() - INTERVAL '24 hours'
            GROUP BY h.video_id, v.title
            ORDER BY total DESC
            LIMIT %s
        """, (k,))
    
    results = cursor.fetchall()
    conn.close()
    
    # Cache the result as JSON with a TTL
    cache_key = f"notebook3:cache:topk:{window}"
    cache_value = json.dumps([
        {"video_id": vid, "title": title, "views": int(views)}
        for vid, title, views in results
    ])
    # TTL of 120 seconds — the cron refreshes every 60s, so we have a buffer
    r.setex(cache_key, 120, cache_value)
    
    return len(results)


def get_top_k(window: str) -> list:
    """
    Client-facing function: read top-K from cache.
    Always fast, never hits the database.
    """
    cache_key = f"notebook3:cache:topk:{window}"
    data = r.get(cache_key)
    if data:
        return json.loads(data)
    return None


# Simulate the cron job running for all windows
print("⏰ Simulating Top-K Cron Job (runs every 60 seconds)")
print("=" * 55)
for window in ["alltime", "last_day", "last_hour"]:
    start = time.time()
    count = precompute_top_k(window, k=10)
    elapsed = (time.time() - start) * 1000
    print(f"  ✅ {window:<12} → cached {count} results ({elapsed:.1f} ms)")

print()
print("Cache entries created with 120-second TTL.")

In [ ]:
# Now simulate client requests reading from cache

print("⚡ Client Reads (from cache only — no database queries!)")
print("=" * 60)

for window in ["alltime", "last_day", "last_hour"]:
    # Measure cache read latency
    times = []
    for _ in range(100):
        start = time.time()
        results = get_top_k(window)
        times.append((time.time() - start) * 1000)
    
    avg_ms = sum(times) / len(times)
    
    print(f"\n  📋 Window: {window} (avg read: {avg_ms:.3f} ms)")
    if results:
        for rank, entry in enumerate(results[:5], 1):
            print(f"     {rank}. {entry['video_id']}: {entry['title'][:30]:<30} ({entry['views']:,} views)")
        if len(results) > 5:
            print(f"     ... and {len(results) - 5} more")
    else:
        print("     (no data — cron hasn't run yet)")

print()
print("🚀 Cache reads are sub-millisecond — easily under the 50ms target!")
print("   Millions of clients can read simultaneously without touching the DB.")

## 🏗️ Putting It All Together

Let's simulate the complete distributed pipeline end-to-end:

1. **Events arrive** → assigned to shards by video ID
2. **Each shard counts** → updates hourly windows in its local DB partition
3. **Cron job runs** → queries each shard's top-K, merges, caches in Redis
4. **Client reads** → always from Redis cache (sub-millisecond)

In [ ]:
# End-to-end simulation of the distributed pipeline

print("🏗️ End-to-End Distributed Top-K Pipeline Simulation")
print("=" * 60)

# --- Step 1: Events arrive and get sharded ---
print("\n📨 Step 1: Events arrive (from Kafka, partitioned by video_id)")
shard_data = [defaultdict(int) for _ in range(NUM_SHARDS)]
for video_id, viewed_at in all_events:
    shard = get_shard(video_id, NUM_SHARDS)
    shard_data[shard][video_id] += 1
print(f"   Distributed {len(all_events):,} events across {NUM_SHARDS} shards")

# --- Step 2: Each shard aggregates locally ---
print("\n📊 Step 2: Each shard computes local top-K")
shard_results = []
for i in range(NUM_SHARDS):
    local_top = sorted(
        shard_data[i].items(), key=lambda x: x[1], reverse=True
    )[:K]
    shard_results.append(local_top)
    print(f"   Shard {i}: top video = {local_top[0][0]} ({local_top[0][1]:,} views)")

# --- Step 3: Cron job merges and caches ---
print("\n🔄 Step 3: Cron merges shard results → Redis cache")
all_candidates = []
for local_top in shard_results:
    all_candidates.extend(local_top)
global_top = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:K]

# Cache in Redis
cache_data = json.dumps([
    {"video_id": vid, "views": count}
    for vid, count in global_top
])
r.setex("notebook3:cache:pipeline_result", 120, cache_data)
print(f"   Cached top {K} in Redis (TTL: 120s)")

# --- Step 4: Client reads from cache ---
print("\n⚡ Step 4: Client reads from cache")
start = time.time()
cached = json.loads(r.get("notebook3:cache:pipeline_result"))
read_time = (time.time() - start) * 1000

print(f"   Read time: {read_time:.3f} ms")
print(f"\n   {'Rank':<6} {'Video ID':<12} {'Views':>8}")
print(f"   {'-'*28}")
for rank, entry in enumerate(cached, 1):
    print(f"   {rank:<6} {entry['video_id']:<12} {entry['views']:>8,}")

print()
print("✅ Complete pipeline: Events → Shards → Merge → Cache → Client")
print("   Client never touches the database. Results served in < 1ms.")

## 👁️ Bonus: Counting *Unique Viewers* with HyperLogLog

Top-K by total views is one question. A harder one: *how many **unique people**
watched each video this hour?* Counting distinct user IDs with a `SET` needs
memory proportional to the number of users — again, too much at scale.

**HyperLogLog (HLL)** estimates the cardinality of a set from a fixed 12 KB of
registers. Redis has it built in: `PFADD` to record, `PFCOUNT` to estimate.

Two things to keep straight, because they are the ones people get wrong:

* Unlike Count-Min Sketch, the HLL error is **two-sided** — it can overestimate
  *or* underestimate. Redis quotes a standard error of **0.81%**, which is a
  standard deviation, not a hard bound.
* Redis stores small HLLs in a **sparse** encoding and switches to the dense
  12 KB layout only once they grow. While sparse, `PFCOUNT` is effectively
  exact — so a demo on a few dozen users proves nothing about HLL's accuracy.
  We therefore run it twice: on our tiny seed data, and at a scale where the
  approximation is actually doing work.


In [ ]:
r = get_redis_client()

# ── Part 1: our seed data — small enough that Redis keeps the HLL sparse ──
users = [f"user_{i}" for i in range(1000)]
random.seed(7)

# Drop anything left over, so re-running this cell doesn't accumulate viewers.
stale = r.keys("notebook3:uniq_*")
if stale:
    r.delete(*stale)

pipe = r.pipeline()
for video_id, _ in all_events:
    user = random.choice(users)
    pipe.sadd(f"notebook3:uniq_set:{video_id}", user)
    pipe.pfadd(f"notebook3:uniq_hll:{video_id}", user)
pipe.execute()

view_totals = defaultdict(int)
for video_id, _ in all_events:
    view_totals[video_id] += 1
top5_vids = [v for v, _ in sorted(view_totals.items(),
                                  key=lambda x: x[1], reverse=True)[:5]]

print("Part 1 — seed data (tens of unique viewers per video):")
print(f"{'Video':<10} {'Exact unique':>13} {'HLL estimate':>14} {'Err':>6} {'HLL bytes':>11}")
print('-' * 60)
for vid in top5_vids:
    exact_u = r.scard(f"notebook3:uniq_set:{vid}")
    hll_u = r.pfcount(f"notebook3:uniq_hll:{vid}")
    err = abs(exact_u - hll_u) / max(exact_u, 1) * 100
    nbytes = r.strlen(f"notebook3:uniq_hll:{vid}")
    print(f"{vid:<10} {exact_u:>13} {hll_u:>14} {err:>5.1f}% {nbytes:>11,}")

print()
print("⚠️  Those errors are ~0% and the keys are a few hundred bytes, not 12 KB.")
print("    That is the SPARSE encoding being exact, not HLL being magic. This")
print("    part of the demo cannot show you the approximation at all.")

# ── Part 2: enough cardinality that the approximation actually matters ────
print()
print("Part 2 — synthetic scale (this is where HLL earns its keep):")
print(f"{'True unique':>12} {'HLL estimate':>14} {'Err':>7} {'HLL bytes':>11} {'exact SET would be':>20}")
print('-' * 70)

CHUNK = 5_000
for true_unique in (1_000, 10_000, 100_000):
    key = f"notebook3:hll_scale:{true_unique}"
    r.delete(key)
    for start in range(0, true_unique, CHUNK):
        batch = [f"viewer_{i}" for i in range(start, min(start + CHUNK, true_unique))]
        r.pfadd(key, *batch)

    est = r.pfcount(key)
    err = abs(est - true_unique) / true_unique * 100
    hll_bytes = r.strlen(key)
    # A SET of the same ids: ~16 bytes of id + ~50 bytes of Redis object/table
    # overhead per member is a conservative floor.
    set_bytes = true_unique * 66
    print(f"{true_unique:>12,} {est:>14,} {err:>6.2f}% {hll_bytes:>11,} {set_bytes:>19,}")

    # 0.81% is a STANDARD error, so allow generous headroom but still fail if
    # the estimator is broken or the key was reused across runs.
    assert err < 5, (
        f"HLL error {err:.2f}% at cardinality {true_unique:,} — far outside the "
        f"0.81% standard error; is the key stale from a previous run?"
    )
    assert hll_bytes <= 12_400, (
        f"HLL key is {hll_bytes:,} bytes at cardinality {true_unique:,}; the dense "
        f"encoding is supposed to cap at ~12 KB regardless of cardinality"
    )

print()
print("💡 Read the last two columns: the HLL flattens out at ~12 KB while the")
print("   exact SET grows linearly — ~6.6 MB at 100K viewers, and it keeps going.")
print("   That constant-memory ceiling is the whole product: 'unique viewers per")
print("   video per day' at YouTube scale, for 12 KB per video per day.")
print()
print("⚠️  And unlike CMS, HLL error goes BOTH ways — check the estimates above")
print("    and you will find some below the true value. Never present an HLL")
print("    number as a lower bound.")

## 🧹 Cleanup

In [ ]:
r = get_redis_client()
keys = r.keys("notebook3:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

## 🌐 Where You See This Exact Pattern in the Wild

The ingredients in this notebook — sharded counters, tumbling buckets, a Redis
leaderboard, a precomputed cache — show up in almost every "top / trending /
popular" feature you've ever seen:

| Product | What they show | Pattern used |
|---|---|---|
| **Twitter / X Trends** | Top hashtags per region | CMS/TopK on the event stream, per-region sharding, 5-minute tumbling windows |
| **Spotify Top Charts** | Most-streamed songs (daily, weekly, all-time) | Kafka → Flink aggregation → precomputed charts in a cache |
| **Reddit r/popular** | Trending posts | Score bucketed per time window + cache |
| **Amazon Best Sellers** | Top products per category | Hourly batch recompute + CDN cache |
| **Google Trends** | Search terms gaining velocity | Sliding-window comparison between recent and baseline counts |
| **YouTube Trending** | Viral videos per country | Exactly what we built in this lab |

The magic is almost never a clever single algorithm — it's the layering:
approximate counting to fit in RAM, tumbling windows to bound computation,
sharding to scale writes, caching to keep reads cheap.


## 📚 Summary

### Key Takeaways

1. **Sharding** splits write load across multiple machines — merging shard-local top-K works
   **only when you shard by the counting key**, so each item's whole count lives on one shard
2. **Tumbling windows** divide time into fixed buckets (e.g., 1 hour) — simplifies time-based aggregation
3. **Redis sorted sets** are purpose-built for leaderboards — ZREVRANGE gives top-K in O(K + log N)
4. **Precomputation + caching** decouples expensive queries from client requests — cron computes, clients read
5. **Batching** reduces write pressure — aggregate counts before writing to the database
6. **Sketches merge; top-K lists do not** — Count-Min Sketches with the same dimensions and
   seeds add element-wise into the exact sketch you'd have built from the combined stream.
   A top-K list has already discarded everything below its local cut, so a broadly-popular
   "slow burn" that is #1 globally and top-K nowhere locally vanishes in the merge
7. **HyperLogLog answers a different question** — unique viewers, not total views, in a fixed
   ~12 KB. Its error is **two-sided** (~0.81% standard error), unlike CMS's one-sided overcount

### The Complete Architecture

```
  View Events → Kafka (partitioned by video_id)
       │
       ├──▶ Shard 0: Consumer → DB Shard 0
       ├──▶ Shard 1: Consumer → DB Shard 1
       └──▶ Shard N: Consumer → DB Shard N
                                    │
                              Cron (every 1 min)
                                    │
                              ┌─────▼─────┐
                              │   Redis    │
                              │  (cache +  │
                              │   ZSETs)   │
                              └─────┬─────┘
                                    │
                              GET /top-k
                              (< 1ms)
```

### Interview Tips

- Start simple (single DB + SQL) → add complexity as needed
- Acknowledge bottlenecks as you go — don't wait for the interviewer to point them out
- Tumbling windows are easier than sliding windows — propose them first, and say out loud that
  they undercount early in a bucket and jump at the boundary
- Say which key you shard by *before* you claim the shard-local top-K merge is correct
- CMS + Heap is great for approximate top-K if the interviewer allows it
- Precompute + cache is the key insight for meeting the <50ms query requirement

### Further Reading

- [Hello Interview — Top-K Problem Breakdown](https://www.hellointerview.com/learn/system-design/problem-breakdowns/top-k)
- [Data Structures for Big Data](https://www.hellointerview.com/learn/system-design/deep-dives/data-structures-for-big-data)
- [Scaling Reads Pattern](https://www.hellointerview.com/learn/system-design/04-patterns/scaling-reads)
- [Scaling Writes Pattern](https://www.hellointerview.com/learn/system-design/04-patterns/scaling-writes)